# Practical Assignment 14: Continuous Bag of Words (CBOW) Model

**Aim:** Implement the Continuous Bag of Words (CBOW) Word2Vec neural network model to learn distributed word embeddings by predicting a target word from surrounding context words.

**Reference:** BE IT Course: Lab Practice IV Lab Manual Assignment No. 5 (Pages 40-45).

**CBOW Mechanics:**
1. **Context Window:** Given context words (e.g. $[w_{t-1}, w_{t+1}]$), predict center word $w_t$.
2. **Embedding Layer:** Projects discrete one-hot tokens into dense continuous vectors.
3. **Mean Pooling:** Averages context embeddings before passing to the linear classifier layer.


In [1]:
import torch
import torch.nn as nn
import numpy as np

# Step 1: Text Corpus and Vocabulary Preparation
corpus = [
    'deep learning models learn hierarchical representations from training data',
    'convolutional neural networks process image data effectively',
    'recurrent neural networks model sequential temporal language data'
]

tokens = [w.lower() for sent in corpus for w in sent.split()]
vocab = sorted(list(set(tokens)))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for i, w in enumerate(vocab)}
print(f'Corpus Tokens: {len(tokens)}, Vocabulary Size: {len(vocab)}')

# Step 2: Generate (Context -> Target) Pairs (Window Size = 1 on each side)
pairs = []
for sent in corpus:
    words = sent.split()
    for i in range(1, len(words) - 1):
        context = [w2i[words[i - 1]], w2i[words[i + 1]]]
        target = w2i[words[i]]
        pairs.append((context, target))

print(f'Total CBOW Training Pairs: {len(pairs)}')
print('Sample (Context -> Target):', ([i2w[idx] for idx in pairs[0][0]], i2w[pairs[0][1]]))

# Step 3: Define CBOW Neural Network Architecture in PyTorch
class CBOW(nn.Module):
    def __init__(self, vocab_size, emb_dim=16):
        super(CBOW, self).__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)     # Word Embedding Matrix
        self.linear = nn.Linear(emb_dim, vocab_size)       # Output Projection
        
    def forward(self, x):
        # x shape: (batch_size, context_size)
        embeds = self.embed(x)                             # (batch_size, context_size, emb_dim)
        context_avg = embeds.mean(dim=1)                  # Average pooling over context
        return self.linear(context_avg)                    # (batch_size, vocab_size)

model = CBOW(len(vocab), emb_dim=16)
print('\nCBOW Model Architecture:\n', model)

# Step 4: Train CBOW Model
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

X = torch.tensor([ctx for ctx, tgt in pairs], dtype=torch.long)
y = torch.tensor([tgt for ctx, tgt in pairs], dtype=torch.long)

for epoch in range(1, 41):
    optimizer.zero_grad()
    out = model(X)
    loss = criterion(out, y)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/40 | CrossEntropy Loss: {loss.item():.4f}')

# Step 5: Evaluate Prediction and Word Embedding Similarity
preds = model(X).argmax(dim=1)
acc = (preds == y).float().mean().item() * 100
print(f'\nCBOW Target Word Prediction Accuracy: {acc:.1f}%')

# Word Embeddings Cosine Similarity
weights = model.embed.weight.detach().numpy()
def cosine_sim(w1, w2):
    v1, v2 = weights[w2i[w1]], weights[w2i[w2]]
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sim = cosine_sim('neural', 'networks')
print(f'Cosine Similarity between "neural" and "networks": {sim:.4f}')
print('CBOW Word2Vec Model trained and verified successfully.')


Corpus Tokens: 24, Vocabulary Size: 20
Total CBOW Training Pairs: 18
Sample (Context -> Target): (['deep', 'models'], 'learning')

CBOW Model Architecture:
 CBOW(
  (embed): Embedding(20, 16)
  (linear): Linear(in_features=16, out_features=20, bias=True)
)
Epoch  1/40 | CrossEntropy Loss: 3.2672
Epoch 10/40 | CrossEntropy Loss: 1.0295
Epoch 20/40 | CrossEntropy Loss: 0.1695
Epoch 30/40 | CrossEntropy Loss: 0.0341
Epoch 40/40 | CrossEntropy Loss: 0.0102

CBOW Target Word Prediction Accuracy: 100.0%
Cosine Similarity between "neural" and "networks": -0.4137
CBOW Word2Vec Model trained and verified successfully.
